# CausalDiscovery：Mistral-7B 因果检测与抽取基线

与 KAPipe notebook 一样，读取项目清洗后的数据，保存逐条预测，调用同一个 `src.evaluator.Evaluator`。

**固定配置：Mistral-7B-Instruct-v0.3 · Detection = FICL · Extraction = CoT · bitsandbytes 4-bit。**
FICL/CoT 使用作者仓库中现成的 prompt 和示例，不根据本项目 test 结果调 prompt；CoT 指作者名为 `chain_of_thought` 的模板，要求静默分析并输出 JSON，不需要开启 LM Studio 的 thinking 开关。

来源：[Anuyah et al., Benchmarking LLMs for Pairwise Causal Discovery in Biomedical and Multi-Domain Contexts](https://arxiv.org/abs/2601.15479)；[作者代码](https://github.com/sydneyanuyah/CausalDiscovery)。论文署名会议年份为 2025，arXiv 上传年份为 2026。报告可标为 **Anuyah et al. (2025) — Mistral-7B (adapted)**；PCD 是任务简称。

**接入方式：原始 text → 作者检测 prompt → 仅对预测正例运行作者抽取 prompt → 原始输出转换 → 全样本 evaluator。**
作者原实验独立运行 detection 和 gold-positive extraction；本 notebook 将它们串联用于项目的端到端比较。这是流程适配，不应称为作者原表格的完全复现。

默认每个数据集只跑前 10 条 smoke；全量关闭。首次实际推理会从 Hugging Face 下载官方权重并在加载时量化，也可以在下方指定已经下载好的 **Hugging Face 模型目录**；GGUF 文件不能用于这里的 Transformers 加载器。

In [ ]:
from pathlib import Path
import importlib.metadata
import logging
import os
import sys

# 选择 Python (Master_thesis) kernel；可从项目根目录或 notebooks 目录启动。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "evaluator.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请在 Master thesis 项目内启动 notebook。")
if Path(sys.prefix).name.lower() != "master_thesis":
    raise RuntimeError(f"当前解释器为 {sys.executable}，请切换到 Master_thesis kernel。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("causal_discovery_notebook")

SOURCE_DIR = ROOT / "reference code from related work" / "CausalDiscovery-main" / "CausalDiscovery-main"
MODEL_NAME_OR_PATH = "mistralai/Mistral-7B-Instruct-v0.3"
# 统一放在项目已忽略的本地缓存中；有既定 HF_HOME 时保留它。
os.environ.setdefault("HF_HOME", str(ROOT / "outputs" / "cache" / "huggingface"))

DATASET_NAMES = ["cnc_sft_test", "li", "ade", "politicause"]
BATCH_SIZE = 4
PRIMARY_METRIC = None  # 使用 evaluator 的数据集默认值，同时报告 Strict 和 Anchor。
RUN_ID = "mistral7b_ficl_cot_v1"
OUTPUT_DIR = ROOT / "results" / "eval_report" / "causal_discovery_baseline"
REUSE_COMPLETED_STAGES = True

INSTALL_DEPENDENCIES = False
RUN_SMOKE = True
EVAL_SAMPLE_N = 10
RUN_FULL_EVAL = False
FULL_EVAL_SAMPLE_N = None

logger.info("项目：%s；解释器：%s", ROOT, sys.executable)

## 1. 环境准备（首次使用）

准备本 notebook 时，本机 RTX 4090 有 24 GB 显存，但 `Master_thesis` 中 PyTorch 为 CPU 版本，且未安装 accelerate / bitsandbytes。先运行下方安装单元，再 **Restart Kernel**，将 `INSTALL_DEPENDENCIES` 改回 `False` 后从头执行。

安装单元默认关闭；开启后会安装 CUDA 12.8 版 PyTorch 2.10.0 和量化依赖。如果当前环境已装 torchvision/torchaudio，同步到对应版本，避免二进制版本不匹配。此版本选择是本机运行环境配置，作者没有提供锁定的完整依赖环境；实际版本会写入每次运行的 manifest。安装前会保存 `pip freeze` 快照。

官方依据：[PyTorch CUDA 12.8 wheels](https://download.pytorch.org/whl/cu128/torch/)、[bitsandbytes 安装说明](https://huggingface.co/docs/bitsandbytes/main/en/installation)。安装 PyTorch 后需要重启 kernel，不能继续使用已载入的 CPU 版模块。

In [ ]:
import subprocess

if INSTALL_DEPENDENCIES:
    if "torch" in sys.modules:
        raise RuntimeError("torch 已被当前 kernel 导入；请先 Restart Kernel，只运行配置与安装单元。")
    env_snapshot = ROOT / "outputs" / "cache" / "causal_discovery" / "environment_before_install.txt"
    env_snapshot.parent.mkdir(parents=True, exist_ok=True)
    frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                            check=True, capture_output=True, text=True)
    env_snapshot.write_text(frozen.stdout, encoding="utf-8")
    torch_packages = ["torch==2.10.0"]
    for package, target in [("torchvision", "0.25.0"), ("torchaudio", "2.10.0")]:
        try:
            importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            continue
        torch_packages.append(f"{package}=={target}")
    subprocess.run([sys.executable, "-m", "pip", "install", *torch_packages,
                    "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "accelerate", "bitsandbytes",
                    "transformers>=4.42", "sentencepiece", "scikit-learn", "pandas", "tqdm"], check=True)
    raise RuntimeError("安装已完成。请 Restart Kernel，将 INSTALL_DEPENDENCIES 改回 False，再从头运行。")
else:
    logger.info("跳过安装；下方环境检查不会加载或下载模型。")

In [ ]:
import json
import pandas as pd
import torch
from IPython.display import Markdown, display
from src.data_io import DATASET_FILES, load_dataset
from src.evaluator import primary_metric_for_dataset
from src.causal_discovery_baseline import (
    AuthorRunner, RunConfig, build_prompts, load_author_templates,
    run_causal_discovery_baseline, software_versions,
)

CONFIG = RunConfig(model_name_or_path=MODEL_NAME_OR_PATH, batch_size=BATCH_SIZE)
TEMPLATES = load_author_templates(SOURCE_DIR)
versions = software_versions()
display(pd.DataFrame([{"Package": name, "Version": version or "NOT INSTALLED"}
                      for name, version in versions.items()]))
CUDA_AVAILABLE = torch.cuda.is_available()
logger.info("CUDA 可用：%s；torch CUDA：%s", CUDA_AVAILABLE, torch.version.cuda)
if CUDA_AVAILABLE:
    logger.info("GPU：%s；总显存 %.1f GiB", torch.cuda.get_device_name(0),
                torch.cuda.get_device_properties(0).total_memory / 1024**3)
else:
    logger.warning("目前只能检查数据和 prompt；实际推理前需完成上方 CUDA 环境安装。")
display(pd.DataFrame([
    {"Stage": "Detection", "Prompt": CONFIG.detection_prompt,
     "Quantization": "bitsandbytes 4-bit; author's defaults (FP4)", "Max new tokens": 512},
    {"Stage": "Extraction", "Prompt": CONFIG.extraction_prompt,
     "Quantization": "bitsandbytes NF4 + double quant; FP16 compute", "Max new tokens": 256},
]))


def require_inference_environment() -> None:
    '''正式推理前检查 CUDA 和量化依赖，给出可执行的下一步。'''
    missing = [name for name in ("accelerate", "bitsandbytes") if not versions[name]]
    if not CUDA_AVAILABLE or missing:
        raise RuntimeError(f"推理环境未就绪：CUDA={CUDA_AVAILABLE}，缺少 {missing}。"
                           "请先执行环境安装单元并重启 kernel。仅检查 notebook 时可关闭 RUN_SMOKE。")
    import accelerate
    import bitsandbytes
    logger.info("量化依赖已导入：accelerate=%s，bitsandbytes=%s", accelerate.__version__, bitsandbytes.__version__)

## 2. 数据与 prompt 核对

使用与现有正式结果相同的数据入口：CNC 固定 test (`cnc_sft_test`)、Li 全数据集、ADE 当前 test、PolitiCAUSE 当前 test。下表统计只用于核对；gold 标签与关系不会进入模型 prompt。

输入适配仅将 `text` 映射为作者 CSV 的 `sentence`。ID 用于对齐输出，不加入 prompt。不应用本项目 RAG，也不添加 chat template，不增删作者示例。

In [ ]:
if EVAL_SAMPLE_N is None or EVAL_SAMPLE_N < 1:
    raise ValueError("Smoke 的 EVAL_SAMPLE_N 必须是正整数。")
if FULL_EVAL_SAMPLE_N is not None and FULL_EVAL_SAMPLE_N < 1:
    raise ValueError("FULL_EVAL_SAMPLE_N 必须为 None 或正整数。")

DATASETS = {name: load_dataset(name) for name in DATASET_NAMES}
dataset_rows = []
for name, samples in DATASETS.items():
    dataset_rows.append({
        "Dataset": name, "Input": DATASET_FILES[name], "N": len(samples),
        "Positive": sum(bool(s["has_causal"]) for s in samples),
        "Gold pairs": sum(len(s.get("relations", [])) for s in samples),
        "Primary extraction metric": PRIMARY_METRIC or primary_metric_for_dataset(name),
    })
display(pd.DataFrame(dataset_rows))

preview_sample = DATASETS[DATASET_NAMES[0]][0]
for stage in ("detection", "extraction"):
    prompt = build_prompts([preview_sample], stage, TEMPLATES, CONFIG)[0]
    display(Markdown(f"### {stage.title()} — 原始模板预览"))
    display(Markdown("```text\n" + prompt + "\n```"))
logger.info("4 份作者源码的 SHA-256 已读取；实际运行会将它们写入 manifest。")

## 3. 小样本运行

每个数据集先取前 `EVAL_SAMPLE_N` 条，不按 gold 选择样本。检测到因果后才运行抽取，随后所有样本一起进入 evaluator；漏检、误检与格式错误不会从评估集合删除。

两个阶段沿用作者不同的 4-bit 加载配置，阶段切换时释放并重新加载模型；因此单次小样本也会有模型加载开销。作者默认 batch 为 64，这里设为 4 适配单卡；采样结果可能随 batch 和软件版本变化。抽取保留 `do_sample=True, top_k=40, temperature=0.7, max_new_tokens=256`，输入截断上限 1024；检测保留作者 pipeline 参数，其余生成默认值随模型/库读取并记录。

In [ ]:
def result_row(result: dict) -> dict:
    '''汇总统一 evaluator 的两种抽取口径，避免只展示 detected-only。'''
    report = result["report"]
    metric = report["extraction"]["primary_metric"]
    extraction = report["extraction"][metric]
    diagnostics = report["baseline"]["diagnostics"]
    return {
        "Dataset": result["dataset"], "N": result["n_samples"],
        "Detection F1": report["detection"]["f1"], "Extraction metric": metric,
        "Extraction F1 (all)": extraction["all_samples"]["f1"],
        "Extraction F1 (detected-only)": extraction["detected_only"]["f1"],
        "Detection parse errors": diagnostics["invalid_detection_outputs"],
        "Extraction parse errors": diagnostics["invalid_extraction_outputs"],
        "Report": str(result["paths"]["report_md"]),
    }


def run_selected_datasets(phase: str, sample_n: int | None) -> list[dict]:
    '''复用作者 runner，并将每个数据集的完整结果保存到独立目录。'''
    require_inference_environment()
    results = []
    for dataset in DATASET_NAMES:
        samples = DATASETS[dataset] if sample_n is None else DATASETS[dataset][:sample_n]
        logger.info("开始 %s / %s：%s 条", phase, dataset, len(samples))
        runner = AuthorRunner(SOURCE_DIR, CONFIG)
        result = run_causal_discovery_baseline(
            samples=samples, runner=runner, dataset=dataset,
            output_dir=OUTPUT_DIR / phase / dataset,
            run_name=f"{RUN_ID}_{dataset}_n{len(samples)}",
            primary_metric=PRIMARY_METRIC,
            reuse_completed_stages=REUSE_COMPLETED_STAGES,
        )
        results.append(result)
        display(Markdown("```text\n" + result["formatted_report"] + "\n```"))
    return results


smoke_results = []
if RUN_SMOKE:
    smoke_results = run_selected_datasets("smoke", EVAL_SAMPLE_N)
    display(pd.DataFrame([result_row(result) for result in smoke_results]))
else:
    logger.info("Smoke 已关闭；未加载模型。")

## 4. 全量评估

确认 smoke 的原始输出、检测标签和多对 span 对应后，在顶部设置 `RUN_FULL_EVAL=True`、`FULL_EVAL_SAMPLE_N=None`，重新运行配置单元与本单元。保留其余模型、prompt 和生成设置；正式结果与 smoke 保存到不同目录。

若显存不足，可以减小顶部 `BATCH_SIZE` 后从配置单元向下执行，并修改 `RUN_ID`。这会形成新的运行记录；不要在同一运行名下混合配置。

In [ ]:
full_results = []
if RUN_FULL_EVAL:
    full_results = run_selected_datasets("full", FULL_EVAL_SAMPLE_N)
    display(pd.DataFrame([result_row(result) for result in full_results]))
else:
    logger.info("全量评估已关闭。开启方式：RUN_FULL_EVAL=True，FULL_EVAL_SAMPLE_N=None。")

## 5. 输出与复核

每个数据集目录保存：

| 文件后缀 | 内容 |
|---|---|
| `.input.csv` | 全部输入的 id 与 sentence，不含 gold |
| `.extraction_input.csv` | 仅检测预测正例的 id 与 sentence |
| `.detection.raw.jsonl` / `.extraction.raw.jsonl` | 逐样本模型原始续写与耗时 |
| `.predictions.jsonl` | evaluator 格式的 has_causal、triples 和解析错误 |
| `.report.json` / `.report.md` | Detection、Strict/Anchor 的 all-samples 与 detected-only 结果 |
| `.manifest.json` | 原始输入与源码哈希、模型、prompt、软件版本和实际量化/生成配置 |

`REUSE_COMPLETED_STAGES=True` 只复用**完整阶段**，并核对输入、源码、配置和依赖版本。若中断在阶段中间，该阶段从头重跑；已完成的检测可直接复用。这样避免在没有保存随机数状态时拼接抽取阶段的采样序列。CSV/原始输出先保存；中断后已写入的内容仍可检查。

输出转换保留 cause→effect 方向、编号配对与重复预测；非原文 span 不被自动修正或丢弃，而是交由相同 evaluator 评分并统计。检测格式错误记录为 `detection_label=null`，由于现有 evaluator 需要布尔值，传入 `has_causal=False`、空 triples；它不是有效的 noncausal 输出，错误数必须随结果报告。检测为正但抽取失败时，仍保留 `has_causal=True`。不会补问模型、重试生成来挑选更好的答案。

复用了作者 `quantize_4bit` / `run_llm_batch`；检测复用其加载与 pipeline 调用参数。作者抽取脚本通过完整字符串前缀删除 prompt，在 tokenizer 归一化或输入截断时可能失效。本适配只在解码输出处按输入 token 长度切出续写，防止把 prompt 内的示例 JSON 当预测；原提示词、tokenizer 输入和生成参数保持不变。

**结果解释：** all-samples 是端到端抽取结果；detected-only 仅在 gold 与预测均为正的样本上比较 span，仍不能当作作者 gold-positive extraction 原表格的一对一复现。论文对比应同时列出 detection 与 all-samples extraction，并注明串联适配及本机量化/软件配置。

In [ ]:
# 查看最近一次结果的前几条原始输出；不重新调用模型。
recent_results = full_results or smoke_results
if recent_results:
    first = recent_results[0]
    display(pd.DataFrame([{"File": name, "Path": str(path)} for name, path in first["paths"].items()]))
    for stage in ("detection", "extraction"):
        raw_path = first["paths"][f"{stage}_raw"]
        with raw_path.open(encoding="utf-8") as file:
            rows = [json.loads(line) for _, line in zip(range(3), file)]
        display(Markdown(f"### {stage.title()} — 原始输出（前 3 条）"))
        display(pd.DataFrame(rows))
else:
    logger.info("尚未生成模型预测。预期输出根目录：%s", OUTPUT_DIR)